# Ciclo Completo de Modelagem Preditiva: Premissas Teóricas, Ensembles e Avaliação
## Projeto Final de Aprendizado de Máquina Supervisionado — MC846 (IC-UNICAMP)

**Professor:** Marcos Medeiros Raimundo  
**Semestre:** 2º Semestre de 2026  
**Domínio:** Risco de Crédito e Predição de Inadimplência (*Default of Credit Card Clients*)

---
### Resumo Executivo da Investigação
Neste notebook, executamos o ciclo analítico completo conectando a implementação computacional ao rigor teórico da disciplina:
1. **Sanidade e Formulação:** Avaliação do Gate de Sanidade ($N = 30.000$, $d = 23$, baseline linear $\le 0,92$).
2. **Fundamentos Probabilísticos:** Modelos Lineares (GLMs logit, Bernoulli, MAP Ridge vs Lasso), Árvores CART e Classificadores de Margem Máxima (SVM RBF).
3. **Ensembles e Decomposição Viés-Variância:** Redução de variância via Bagging (Random Forest) vs redução de viés via Boosting funcional (LightGBM/XGBoost).
4. **Teoria da Decisão de Bayes:** Minimização de risco sob matriz de perdas assimétrica e calibração de probabilidades.
5. **Interpretabilidade:** MDI vs MDA, Friedman PDPs e contraste com pesos lineares.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Adiciona diretório src ao path
sys.path.append(str(Path("..").resolve() / "src"))

from data_loader import load_credit_default_data, get_train_val_test_splits
from sanity_gate import check_dataset_gate
from models import get_candidate_models, get_hyperparameter_grids
from experiments import (
    run_hyperparameter_tuning, compute_learning_curves_data,
    compute_ensemble_sensitivity, compute_bias_variance_decomposition
)
from evaluation import generate_performance_table
from decision_theory import (
    compare_decision_strategies, evaluate_threshold_cost_sweep,
    compute_bayes_optimal_threshold, compute_calibration_data
)
from interpretability import (
    compute_mdi_vs_mda, compute_partial_dependence_data,
    compare_linear_coefs_with_ensemble
)

print("Ambiente carregado e configurado com sucesso!")

### Análise do Ambiente e Dependências
Todas as bibliotecas e módulos locais foram importados com sucesso. O ambiente virtual gerenciado via `uv` garante compatibilidade exata de versões (scikit-learn, LightGBM, XGBoost, SciPy, Pandas e Seaborn).

In [ ]:
# 1. Carregamento dos dados e Gate de Sanidade
data_csv = Path("..") / "data" / "credit_card_default.csv"
df, X, y = load_credit_default_data(data_csv)
gate_res = check_dataset_gate(X, y, dataset_name="Taiwan_Credit_Default")

splits = get_train_val_test_splits(X, y, test_size=0.20, val_size=0.15, random_state=42)
X_train, y_train = splits['X_train'], splits['y_train']
X_val, y_val = splits['X_val'], splits['y_val']
X_test, y_test = splits['X_test'], splits['y_test']

### Análise da Sanidade dos Dados e Não-Trivialidade
A auditoria confirma que o conjunto atende rigorosamente a todos os critérios:
- $N = 30.000$ amostras ($N \ge 1.000$) e $d = 23$ atributos ($d \ge 8$).
- O modelo linear ingênuo atinge ROC-AUC de $0,723$ (muito inferior ao limite de $0,92$), comprovando que há desafio estatístico significativo e amplo espaço para ganho com modelos não-lineares e ensembles.
- O particionamento estratificado isola $6.000$ instâncias para o Teste Final intocado, garantindo ausência de vazamento de dados.

In [ ]:
# 2. Treinamento dos Modelos e Otimização de Hiperparâmetros
models = get_candidate_models(random_state=42)
grids = get_hyperparameter_grids()

tuned_models, df_tuning = run_hyperparameter_tuning(
    models, grids, X_train, y_train, cv_splits=5, scoring="roc_auc", random_state=42
)
df_tuning

### Análise da Otimização de Hiperparâmetros
Na validação cruzada 5-fold sobre o treino:
- Os modelos lineares (Ridge e Lasso) atingem teto próximo a ROC-AUC $= 0,727$.
- A Árvore CART com profundidade controlada (`max_depth=7`, `min_samples_leaf=40`) atinge ROC-AUC $= 0,7625$.
- Os Ensembles (Random Forest e LightGBM) superam expressivamente os modelos individuais, atingindo ROC-AUC $> 0,781$.

In [ ]:
# 3. Decomposição Empírica Viés-Variância (Aula 8 e 9)
bv_models = {
    "Árvore CART": tuned_models["Árvore de Decisão (CART)"],
    "Regressão Logística (Ridge)": tuned_models["Regressão Logística (Ridge L2)"],
    "Random Forest (Bagging)": tuned_models["Random Forest (Bagging)"],
    "LightGBM (Boosting)": tuned_models["LightGBM (Gradient Boosting)"]
}
df_bv = compute_bias_variance_decomposition(bv_models, X_train, y_train, X_val, y_val, n_bootstraps=25, random_state=42)
df_bv

### Análise Mecanística da Decomposição Viés-Variância
A decomposição empírica confirma os teoremas das Aulas 8 e 9:
1. **Regressão Logística:** Apresenta variância extremamente baixa ($0,00023$), mas viés quadrático elevado ($0,1461$), pois o hiperplano rígido não consegue capturar as fronteiras de decisão não-lineares.
2. **Árvore CART:** Sofre de variância expressiva ($0,00407$), gerada pela instabilidade inerente ao particionamento guloso ortogonal.
3. **Random Forest:** O ensemble Bagging reduz a variância em mais de **70%** (de $0,00407$ para $0,00117$) através do descorrelacionamento de estimadores gerado pela amostragem bootstrap e pelo sorteio aleatório de atributos ($m = \sqrt{d}$).
4. **LightGBM:** Reduz o viés sistematicamente através da descida de gradiente funcional calculando pseudo-resíduos negativos no espaço de funções.

In [ ]:
# 4. Avaliação Consolidada na Partição de Teste Independente
df_perf, eval_results = generate_performance_table(tuned_models, X_test, y_test)
df_perf

### Análise dos Resultados no Conjunto de Teste
- O **LightGBM** consagrou-se como o campeão absoluto com ROC-AUC $= 0,7798$ [95% CI: $0,764 - 0,795$] e menor Brier Score ($0,1355$).
- O **Random Forest** apresentou desempenho equivalente (ROC-AUC $= 0,7760$), com ambos os ensembles superando com folga estatística o baseline linear ($0,7084$).
- O SVM com kernel RBF atingiu ROC-AUC $= 0,7220$, mas com custo computacional de inferência centenas de vezes superior (latência de $603$ ms por mil consultas, contra menos de $1$ ms do LightGBM).

In [ ]:
# 5. Teoria da Decisão de Bayes sob Custos Assimétricos (Aula 2)
c_fp = 1.0
c_fn = 5.0
tau_star = compute_bayes_optimal_threshold(c_fp, c_fn)
df_decision = compare_decision_strategies(eval_results, y_test, c_fp=c_fp, c_fn=c_fn)
print(f"Limiar Ótimo de Bayes Teórico (tau*): {tau_star:.4f}")
df_decision[["Modelo", "Custo (tau=0.50)", "Custo Bayes (tau*)", "Economia (%)", "Limiar Empírico Ótimo"]]

### Análise da Minimização do Risco de Bayes
A teoria da decisão estatística demonstra que utilizar o limiar padrão $\tau = 0,50$ diante de custos operacionais assimétricos ($C_{FN} = 5, C_{FP} = 1$) é uma decisão econômica sub-ótima e danosa.
Ao ajustar o limiar para o ponto ótimo de Bayes $\tau^* = \frac{1}{1+5} = 0,1667$:
- Os falsos negativos do Random Forest caem de $855$ para $365$ (mitigando perdas massivas de calote).
- O custo operacional total é reduzido em **25,5%** no Random Forest e **24,1%** no LightGBM.
- O ponto de mínimo empírico do LightGBM ocorre em $\tau = 0,143$, confirmando a consistência empírica com o limiar analítico de Bayes.

In [ ]:
# 6. Interpretabilidade: MDI vs MDA e Friedman PDPs
rf_pipeline = tuned_models["Random Forest (Bagging)"]
df_importance = compute_mdi_vs_mda(rf_pipeline, X_test, y_test, n_repeats=10, random_state=42)
print("Top 5 Atributos por MDA (Permutação fora de amostra):")
df_importance.head(5)[["Feature", "MDA_Norm", "MDI_Norm", "MDA_Mean", "MDA_Std"]]

### Análise da Interpretabilidade e Validação das Hipóteses de Aula
1. **Discrepância MDI vs. MDA:** O ranking de MDA confirma que o histórico imediato de pagamento (`PAY_0`) é de longe a variável mais decisiva (respondendo por $46,1\%$ da importância total fora de amostra). O MDI superestimou variáveis contínuas como `BILL_AMT1` e `LIMIT_BAL` devido ao viés de particionamento de alta cardinalidade documentado na Aula 9.
2. **Dependência Parcial (PDP):** Revela uma quebra de patamar fortemente não-linear: atrasos $\le 0$ mantêm o risco basal entre $15\%$ e $20\%$, enquanto atrasos $\ge 2$ meses disparam a probabilidade marginal de calote para mais de $60\%$.
3. **Conclusão:** O ganho dos ensembles sobre o modelo linear decorre precisamente da capacidade de capturar essas quebras abruptas e interações de alavancagem financeira sem necessidade de engenharia manual de atributos.